In [17]:
from google.colab import drive
drive.mount('/content/drive')

project_path = "/content/drive/MyDrive/ML Projects/Airfoil/"

train = pd.read_csv(project_path + "train (1).csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
%%writefile feature_engineering.py
import numpy as np

def feature_engineering(df):
    df = df.copy()

    df["freq_vel_interaction"] = df["frequency"] * df["free-stream-velocity"]
    df["strouhal"] = (df["frequency"] * df["chord-length"]) / df["free-stream-velocity"]
    df["strouhal_squared"] = df["strouhal"] ** 2
    df["strouhal_cubed"] = df["strouhal"] ** 3
    df["strouhal_angle"] = df["strouhal"] * df["attack-angle"]

    df["log_strouhal_angle"] = np.log1p(np.maximum(df["strouhal_angle"], -0.999999))

    df["velocity5_thickness"] = (df["free-stream-velocity"] ** 5) * df["displacement-thickness"]
    df["angle_vel_interaction"] = df["attack-angle"] * df["free-stream-velocity"]
    df["angle_freq_interaction"] = df["attack-angle"] * df["frequency"]
    df["angle_chord_interaction"] = df["attack-angle"] * df["chord-length"]

    df["angle_squared"] = df["attack-angle"] ** 2
    df["angle_freq_squared"] = df["attack-angle"] * (df["frequency"] ** 2)

    df.drop(columns=["velocity5_thickness","strouhal_angle","strouhal"], inplace=True)
    return df

Writing feature_engineering.py


In [8]:
%%writefile train_model.py
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

def train_model(X_train, y_train):
    """
    Trains an XGBoost regression model using GridSearchCV and returns:
    - best_estimator_: the fitted model
    - grid: the GridSearchCV object (for accessing CV results)
    - train_r2: R² on the full training dataset
    """

    # Pipeline: Standardize inputs → Train XGBoost model
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBRegressor(objective='reg:squarederror', random_state=1))
    ])

    # Hyperparameters (your tuned ones)
    param_grid = {
        'xgb__n_estimators': [480],
        'xgb__max_depth': [9],
        'xgb__learning_rate': [0.10704067150254538],
        'xgb__subsample': [0.8],
        'xgb__colsample_bytree': [1.0],
        'xgb__reg_alpha': [0.1],
        'xgb__reg_lambda': [10],
        'xgb__min_child_weight': [5],
        'xgb__gamma': [0],
    }

    # Grid search
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='r2',
        cv=12,
        n_jobs=-1,
        verbose=1
    )

    # Fit model
    grid.fit(X_train, y_train)

    # Training R²
    train_r2 = grid.best_estimator_.score(X_train, y_train)

    return grid.best_estimator_, grid, train_r2


Writing train_model.py


In [9]:
import pandas as pd

# 📂 Load data again here
train = pd.read_csv('train (1).csv')
test  = pd.read_csv('test (2).csv')

# Columns
X_cols = [
    'frequency','attack-angle','chord-length',
    'free-stream-velocity','displacement-thickness'
]
y_col = 'scaled-sound-pressure'

# Load raw feature matrices
X_train_raw = train[X_cols]
y_train = train[y_col]

# Use your FE function
from feature_engineering import feature_engineering
X_train = feature_engineering(X_train_raw)


In [12]:
from sklearn.metrics import mean_squared_error

def validate(model, X_train, y_train, test_df=None):

    train_r2 = model.score(X_train, y_train)
    print("🎯 Training R²:", train_r2)

    if test_df is not None and y_col in test_df.columns:
        y_test = test_df[y_col]
        X_test_feat = feature_engineering(test_df[X_cols])
        preds = model.predict(X_test_feat)

        test_r2 = model.score(X_test_feat, y_test)
        rmse = mean_squared_error(y_test, preds, squared=False)

        print("🧪 Test R²:", test_r2)
        print("📉 Test RMSE:", rmse)
    else:
        print("⚠️ No ground truth in test.csv — cannot compute test R².")


In [20]:
# -----------------------------
# TEST validate() IN THIS NOTEBOOK
# -----------------------------

import pandas as pd
from feature_engineering import feature_engineering
from train_model import train_model

# Path to your project folder
project_path = "/content/drive/MyDrive/ML Projects/Airfoil/"

# 1. Load train.csv
train = pd.read_csv(project_path + "train (1).csv")

# Define columns
X_cols = [
    'frequency', 'attack-angle', 'chord-length',
    'free-stream-velocity', 'displacement-thickness'
]
y_col = 'scaled-sound-pressure'

# 2. Prepare training data
X_train_raw = train[X_cols]
X_train = feature_engineering(X_train_raw)
y_train = train[y_col]

# 3. Train model (or load a saved one)
model, grid_result, train_r2 = train_model(X_train, y_train)

# 4. Load test.csv (if it has labels, validate on test)
test_path = project_path + "test (2).csv"
test_df = pd.read_csv(test_path)

# 5. Run validate()
print("\n--- VALIDATION OUTPUT ---\n")
validate(
    model=model,
    X_train=X_train,
    y_train=y_train,
    test_df=test_df  # or None if test has no labels
)


Fitting 12 folds for each of 1 candidates, totalling 12 fits

--- VALIDATION OUTPUT ---

🎯 Training R²: 0.9999773119420516
⚠️ No ground truth in test.csv — cannot compute test R².
